# Eval Loop

In [1]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

In [4]:
test_preds = {
    "tamil" : {
        "y_test" : pd.read_csv("misogyny/tamil/test_with_labels.csv"),
        "y_pred" : {
                "llava" : pd.read_csv("misogyny/tamil/test_pred_llava_tamil.csv", keep_default_na=False),
                "qwen" : pd.read_csv("misogyny/tamil/test_pred_qwen_tamil.csv", keep_default_na=False),
                "qwen3" : pd.read_csv("misogyny/tamil/test_pred_qwen3_tamil.csv", keep_default_na=False),
                "mistral" : pd.read_csv("misogyny/tamil/test_pred_mistral_tamil.csv", keep_default_na=False),
            }
    },
    "malayalam" : {
        "y_test" : pd.read_csv("misogyny/malayalam/test_with_labels.csv"),
        "y_pred" : {
                "llava" : pd.read_csv("misogyny/malayalam/test_pred_llava_malayalam.csv", keep_default_na=False),
                "qwen": pd.read_csv("misogyny/malayalam/test_pred_qwen_malayalam.csv", keep_default_na=False),
                "qwen3" : pd.read_csv("misogyny/malayalam/test_pred_qwen3_malayalam.csv", keep_default_na=False),
                "mistral" : pd.read_csv("misogyny/malayalam/test_pred_mistral_malayalam.csv", keep_default_na=False),
            }
    }
}

In [5]:
for lang, data in test_preds.items():
    print(f"\n{'='*40}\n{lang.upper()}\n{'='*40}")

    # Process y_test
    y_test = data["y_test"].copy()
    y_test['labels'] = y_test['labels'].str.lower().replace({"not-misogyny": 0, "not misogyny": 0, "misogyny": 1})
    y_test = y_test.rename(columns={'fimage_id': 'image_id'})

    for pred, y_pred in data["y_pred"].items():
        test_pred = y_pred.copy()

        # Process y_pred
        test_pred['labels'] = (
            test_pred['labels']
            .str.lower()
            .str.strip()
            .replace({"misogynistic": 1, "na": 0, "nan": 0, "none": 0, "n/a": 0})
            .apply(lambda x: -1 if x not in [0, 1] else x)
        )

        merged = y_test.merge(test_pred, on='image_id', suffixes=('_true', '_pred'))
        try:
            merged['labels_true'] = pd.to_numeric(merged['labels_true'], errors='coerce').astype(int)
            merged['labels_pred'] = pd.to_numeric(merged['labels_pred'], errors='coerce').astype(int)
        except Exception as E:
            print(merged)

        accuracy = accuracy_score(merged['labels_true'], merged['labels_pred'])
        f1 = f1_score(merged['labels_true'], merged['labels_pred'], average='macro')

        print(f"\n  [{pred}]")
        print(f"  Accuracy : {accuracy:.4f}")
        print(f"  Macro F1 : {f1:.4f}")


TAMIL

  [llava]
  Accuracy : 0.7303
  Macro F1 : 0.3773

  [qwen]
  Accuracy : 0.7107
  Macro F1 : 0.2838

  [qwen3]
  Accuracy : 0.6994
  Macro F1 : 0.3248

  [mistral]
  Accuracy : 0.7022
  Macro F1 : 0.3254

MALAYALAM

  [llava]
  Accuracy : 0.6100
  Macro F1 : 0.4123

  [qwen]
  Accuracy : 0.6000
  Macro F1 : 0.3750

  [qwen3]
  Accuracy : 0.6000
  Macro F1 : 0.2660

  [mistral]
  Accuracy : 0.5650
  Macro F1 : 0.2491


In [89]:
merged["labels_pred"].value_counts().reindex([0, 1, -1], fill_value=0)

labels_pred
 0    200
 1      0
-1      0
Name: count, dtype: int64

In [36]:
test.head()

,fimage_id,labels,transcriptions
0,1193,Misogyny,கணவர் செல்லம் சப்பாத்தி சூப்பரா செஞ்சிருக்கடி ...
1,427,not-misogyny,MUTHUMC Omuhu ui *Manager discussing with TL t...
2,1474,Misogyny,VIPER MEMLZ IRUKI EDHOUIAIRUKU
3,766,not-misogyny,In Irugapatru Director Yuvaraj Dhayalan has gi...
4,486,not-misogyny,Dad : யாரையாச்சும் Love பண்ணா சொல்றா கட்டிவைக்...


In [120]:
test_pred.head()

,image_id,labels
0,1262,NaN
1,744,NaN
2,1010,NaN
3,1237,NaN
4,620,NaN


In [37]:
test['labels'] = test['labels'].str.lower().replace("not-misogyny", 0).replace("misogyny", 1)
test_pred['labels'] = test_pred['labels'].str.lower().fillna(0).replace("misogynistic", 1)

In [38]:
test_pred['labels'].unique()

array([0, 1, 'intha naai podra uru biscuit ku',
       'ala patha dummy piece ah iruka ana'], dtype=object)

In [39]:
# Set anything not 0 or 1 to 0
test_pred['labels'] = test_pred['labels'].apply(lambda x: x if x in [0, 1] else -1)
test = test.rename(columns={'fimage_id': 'image_id'})

In [40]:
test.head()

,image_id,labels,transcriptions
0,1193,1,கணவர் செல்லம் சப்பாத்தி சூப்பரா செஞ்சிருக்கடி ...
1,427,0,MUTHUMC Omuhu ui *Manager discussing with TL t...
2,1474,1,VIPER MEMLZ IRUKI EDHOUIAIRUKU
3,766,0,In Irugapatru Director Yuvaraj Dhayalan has gi...
4,486,0,Dad : யாரையாச்சும் Love பண்ணா சொல்றா கட்டிவைக்...


In [41]:
test_pred.head()

,image_id,labels
0,1262,0
1,744,0
2,1010,0
3,1237,0
4,620,0


In [42]:
from sklearn.metrics import accuracy_score, f1_score

# Merge on image_id to align labels
merged = test.merge(test_pred, on='image_id', suffixes=('_true', '_pred'))

merged.head()

,image_id,labels_true,transcriptions,labels_pred
0,1193,1,கணவர் செல்லம் சப்பாத்தி சூப்பரா செஞ்சிருக்கடி ...,0
1,427,0,MUTHUMC Omuhu ui *Manager discussing with TL t...,0
2,1474,1,VIPER MEMLZ IRUKI EDHOUIAIRUKU,0
3,766,0,In Irugapatru Director Yuvaraj Dhayalan has gi...,0
4,486,0,Dad : யாரையாச்சும் Love பண்ணா சொல்றா கட்டிவைக்...,0


In [43]:
merged['labels_true'] = pd.to_numeric(merged['labels_true'], errors='coerce').fillna(0).astype(int)
merged['labels_pred'] = pd.to_numeric(merged['labels_pred'], errors='coerce').fillna(0).astype(int)

# Verify
print(merged['labels_true'].unique())
print(merged['labels_pred'].unique())

[1 0]
[ 0  1 -1]


In [44]:
accuracy = accuracy_score(merged['labels_true'], merged['labels_pred'])
f1 = f1_score(merged['labels_true'], merged['labels_pred'], average='macro')

print(f"Accuracy: {accuracy:.4f}")
print(f"Macro F1: {f1:.4f}")

Accuracy: 0.7303
Macro F1: 0.3773


# Malayalam

In [45]:
test = pd.read_csv("misogyny/malayalam/test_with_labels.csv")
test_pred = pd.read_csv("misogyny/malayalam/test_pred_llava_malayalam.csv")

In [46]:
test['labels'] = test['labels'].str.lower().replace("not misogyny", 0).replace("misogyny", 1)
test_pred['labels'] = test_pred['labels'].str.lower().fillna(0).replace("misogynistic", 1)

print("test:", (~test['labels'].isin([0, 1])).sum())
print("test_pred:", (~test_pred['labels'].isin([0, 1])).sum())

test = test.rename(columns={'fimage_id': 'image_id'})

test: 0
test_pred: 0


In [47]:
test['labels'].unique()

array([0, 1], dtype=object)

In [48]:
test_pred['labels'].unique()

array([0, 1], dtype=object)

In [49]:
from sklearn.metrics import accuracy_score, f1_score

# Merge on image_id to align labels
merged = test.merge(test_pred, on='image_id', suffixes=('_true', '_pred'))

merged.head()

,image_id,labels_true,transcriptions,labels_pred
0,522,0,15000 രൂപയ്ക്ക് പഴയ 800 വാങ്ങി അത് 3ലക്ഷം രൂപയ...,0
1,738,0,\nഈ ചരിത്രം എന്ന് പറയുന്നത് തിരുത്തി എഴുതാനുള്...,0
2,741,0,asianet news\n13.12.2023\n“വിഴിഞ്ഞം തുറമുഖം നാ...,0
3,661,1,മിനി സ്ക്രീനിലെ വെടി എന്നൊക്കെ കേട്ടിട്ടുണ്ടെങ...,0
4,412,1,ആരാടീ അത്? ഞാനില്ലാത്ത നേരത്ത് നീ... നാണമുണ്ട...,0


In [50]:
merged['labels_true'] = pd.to_numeric(merged['labels_true'], errors='coerce').fillna(0).astype(int)
merged['labels_pred'] = pd.to_numeric(merged['labels_pred'], errors='coerce').fillna(0).astype(int)

# Verify
print(merged['labels_true'].unique())
print(merged['labels_pred'].unique())

[0 1]
[0 1]


In [51]:
accuracy = accuracy_score(merged['labels_true'], merged['labels_pred'])
f1 = f1_score(merged['labels_true'], merged['labels_pred'], average='macro')

print(f"Accuracy: {accuracy:.4f}")
print(f"Macro F1: {f1:.4f}")

Accuracy: 0.6100
Macro F1: 0.4123


# QWEN-Tamil

In [19]:
import pandas as pd

test = pd.read_csv("misogyny/tamil/test_with_labels.csv")

test_pred = pd.read_csv("misogyny/tamil/test_pred_qwen_tamil.csv", keep_default_na=False)

In [20]:
test_pred.head()

,image_id,labels
0,1262,NA
1,744,NA
2,1010,NA
3,1237,NA
4,620,NA


In [21]:
test['labels'] = test['labels'].str.lower().replace("not-misogyny", 0).replace("misogyny", 1)

test_pred['labels'] = (
    test_pred['labels']
    .str.lower()
    .str.strip()
    .replace("misogynistic", 1)
    .replace("na", 0)
    .apply(lambda x: -1 if x not in [0, 1] else x)
)

In [22]:
test_pred.head()

,image_id,labels
0,1262,0
1,744,0
2,1010,0
3,1237,0
4,620,0


In [23]:
test_pred['labels'].value_counts().reindex([0, 1, -1], fill_value=0)

labels
 0    350
 1      2
-1      4
Name: count, dtype: int64

In [25]:
from sklearn.metrics import accuracy_score, f1_score

test = test.rename(columns={'fimage_id': 'image_id'})

# Merge on image_id to align labels
merged = test.merge(test_pred, on='image_id', suffixes=('_true', '_pred'))

merged.head()

,image_id,labels_true,transcriptions,labels_pred
0,1193,1,கணவர் செல்லம் சப்பாத்தி சூப்பரா செஞ்சிருக்கடி ...,0
1,427,0,MUTHUMC Omuhu ui *Manager discussing with TL t...,0
2,1474,1,VIPER MEMLZ IRUKI EDHOUIAIRUKU,1
3,766,0,In Irugapatru Director Yuvaraj Dhayalan has gi...,0
4,486,0,Dad : யாரையாச்சும் Love பண்ணா சொல்றா கட்டிவைக்...,0


In [29]:
merged['labels_true'] = pd.to_numeric(merged['labels_true'], errors='coerce').fillna(0).astype(int)
merged['labels_pred'] = pd.to_numeric(merged['labels_pred'], errors='coerce').fillna(0).astype(int)

accuracy = accuracy_score(merged['labels_true'], merged['labels_pred'])
f1 = f1_score(merged['labels_true'], merged['labels_pred'], average='macro')

print(f"Accuracy: {accuracy:.4f}")
print(f"Macro F1: {f1:.4f}")

Accuracy: 0.7107
Macro F1: 0.2838
